# Custom Model sweep — single self-contained notebook

All code is in this notebook (no external `.py` files). It reads intake fields from Snowflake via the notebook's session, merges them into the ThoughtSpot template model, and imports each org's **"Custom Model"**.

**Run top to bottom.** Sections 1–2 define everything; section 3 is your creds; section 4 runs it. Set `DRY_RUN = False` to actually import.

## 1. Dependencies
Add `ruamel.yaml` + `requests` via the **Packages** dropdown, or run this cell.

In [ ]:
import importlib, subprocess, sys
for _imp, _pip in [("ruamel.yaml", "ruamel.yaml"), ("requests", "requests")]:
    try:
        importlib.import_module(_imp)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _pip])
print("deps ready")

## 2. Library
Run each block once (top to bottom — later blocks use names from earlier ones).

### Field types → ThoughtSpot formula expressions
`IntakeFieldInput` is the shape of one Snowflake field row. `field_type_to_model_formula_expr` turns a field type into the TML formula (a Snowflake passthrough that reads the value out of `RESPONSES_JSON`).

In [ ]:
# --- intake_field_types ---
"""Snowflake field metadata merged into model TML formulas.

Port of ``src/tml/intakeFieldTypes.ts``. The TS ``IntakeFieldInput`` interface is
a plain shape; here it's a dataclass with the same four fields.
"""


from dataclasses import dataclass
from typing import Optional


@dataclass
class IntakeFieldInput:
    field_id: str
    field_label: str
    field_type: Optional[str]
    group_name: Optional[str]

# --- field_type_to_expr ---
"""Field type -> ThoughtSpot model TML formula expression.

Port of ``src/tml/fieldTypeToExpr.ts``.

Maps a Submittable form field type (e.g. "text", "number", "date", "textarea")
to a ThoughtSpot model TML formula expression. The expression uses the Snowflake
passthrough operator matching the value's type (``sql_string_op``,
``sql_double_op`` or ``sql_date_time_op``) and casts the JSON ``get()`` result to
a compatible Snowflake type. Anything not explicitly mapped is treated as string.
"""


from typing import Optional, Tuple


def _sql_op_for_field_type(field_type: Optional[str]) -> Tuple[str, str]:
    """Return ``(op, cast)`` for a field type.

    NOTE (faithful port): the TS source lower-cases the type but then compares
    against the mixed-case literal ``"calculatedValue"`` in its switch, so that
    branch never matched and calculatedValue fields fell through to the string
    default. That exact behavior is preserved here so output TML is unchanged.
    """
    t = (field_type or "").strip().lower()
    if t == "number":
        return ("sql_double_op", "::double")
    if t == "calculatedValue":  # never matches (t is lower-cased) -- see note above
        return ("sql_double_op", "::double")
    if t == "date":
        return ("sql_date_time_op", "::timestamp")
    # "text", "textarea", "wysiwyg", "singleselect", unknown, etc.
    return ("sql_string_op", "::string")


def field_type_to_model_formula_expr(
    field_type: Optional[str],
    field_id: str,
    sub_field_id: str,
    logical_table: str,
) -> str:
    """Snowflake passthrough with a qualified logical column
    ``[LogicalTable::RESPONSES_JSON]`` (ThoughtSpot list syntax). Operator and
    cast are chosen from ``field_type``.
    """
    op, cast = _sql_op_for_field_type(field_type)
    list_arg = f"[{logical_table}::RESPONSES_JSON]"
    escaped_id = field_id.replace("\\", "\\\\").replace("'", "\\'")
    inner = f"get({{0}}, '{escaped_id}_{sub_field_id}'){cast}"
    return f'{op} ( "{inner}" , {list_arg} )'

### Serialize TML (YAML)
Parse/serialize ThoughtSpot TML. Serialization double-quotes string values, keeps keys plain, and elides explicit nulls inside `data_panel_column_groups` (what ThoughtSpot's loader expects).

In [ ]:
# --- serialize_tml ---
"""Parse/serialize TML YAML (port of ``src/tml/serializeTml.ts``).

The TS version used the ``yaml`` npm package with ``defaultStringType:
QUOTE_DOUBLE`` and ``defaultKeyType: PLAIN`` (string *values* double-quoted,
keys left plain), then post-processed the text to elide explicit nulls inside
``data_panel_column_groups`` (ThoughtSpot's TML loader rejects ``key: null``
there and wants an implicit-null ``key:``).

We reproduce that exactly with ruamel.yaml: parse to plain dicts/lists, wrap
string values (not keys) in double-quoted scalars, render ``None`` as ``null``,
disable line wrapping, then run the same null-eliding pass over the text.
"""


import io
import re
from typing import Any

from ruamel.yaml import YAML
from ruamel.yaml.scalarstring import DoubleQuotedScalarString


def parse_tml(yaml_text: str) -> Any:
    """Parse ThoughtSpot TML YAML into plain Python objects."""
    yaml = YAML(typ="safe")
    yaml.preserve_quotes = False
    return yaml.load(yaml_text)


def _quote_string_values(value: Any) -> Any:
    """Recursively wrap string *values* in double-quoted scalars, leaving mapping
    keys plain. Mirrors ``defaultStringType: QUOTE_DOUBLE`` + ``defaultKeyType:
    PLAIN``.
    """
    if isinstance(value, dict):
        # Keys stay as plain str; only recurse into values.
        return {k: _quote_string_values(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_quote_string_values(v) for v in value]
    if isinstance(value, str):
        return DoubleQuotedScalarString(value)
    return value


def _represent_none(representer, _data):
    # Match the npm `yaml` package, which emits `null` for None; the elide pass
    # below then strips it inside data_panel_column_groups.
    return representer.represent_scalar("tag:yaml.org,2002:null", "null")


def _elide_explicit_nulls_in_data_panel_groups(yaml_text: str) -> str:
    """Rewrite ``key: null`` -> ``key:`` for lines nested under a
    ``data_panel_column_groups:`` block (port of the TS post-processor).
    """
    lines = yaml_text.split("\n")
    block_indent = -1
    i = 0
    while i < len(lines):
        line = lines[i]
        open_match = re.match(r"^(\s*)data_panel_column_groups:\s*$", line)
        if open_match:
            block_indent = len(open_match.group(1))
            i += 1
            continue
        if block_indent < 0:
            i += 1
            continue

        indent_match = re.match(r"^(\s*)\S", line)
        if not indent_match:
            i += 1
            continue
        line_indent = len(indent_match.group(1))
        if line_indent <= block_indent:
            block_indent = -1
            # Re-examine this same line (it may open another block); mirrors the
            # TS `i--` + loop increment.
            continue
        lines[i] = re.sub(r": null\s*$", ":", line)
        i += 1
    return "\n".join(lines)


def serialize_tml(doc: Any) -> str:
    """Serialize TML as YAML with double-quoted string values, matching the TS output."""
    yaml = YAML()
    yaml.default_flow_style = False
    yaml.width = 1 << 30  # equivalent to the TS `lineWidth: 0` (no wrapping)
    yaml.allow_unicode = True
    yaml.representer.add_representer(type(None), _represent_none)

    buf = io.StringIO()
    yaml.dump(_quote_string_values(doc), buf)
    return _elide_explicit_nulls_in_data_panel_groups(buf.getvalue())

### Merge fields into the model TML
`merge_intake_formulas_into_model_tml` appends each field to `model.formulas` and `model.columns`, groups columns by program name, and registers the DATA_PANEL column groups.

In [ ]:
# --- merge_formulas ---
"""Merge intake field metadata into a base model TML document.

Port of ``src/tml/mergeIntakeFormulasIntoModelTml.ts``. Appends to
``model.formulas`` / ``model.columns`` (each formula bound to ``logical_table``),
tags generated columns with ``data_panel_column_groups`` by program name, and
registers every program group under the model's ``column_groups`` DATA_PANEL
section. Safe to call repeatedly to accumulate fields from multiple tables. The
input doc is deep-copied before mutation.
"""


import copy
from typing import Any, Dict, Iterable, List, Optional


# Field type -> the sub-field ids that expand into columns.
_FIELD_SUBFIELD_MAPPINGS: Dict[str, List[str]] = {
    "vote": ["label"],
    "textarea": ["text"],
    "multiselect": ["label"],
    "date": ["startDate", "date", "endDate"],
    "address": ["city", "state", "postal", "line1", "line2", "country"],
    "lookup": [
        "CollegeCampusCity", "CollegeFinancialAddress1", "CollegeFinancialAddress2",
        "CollegeFinancialCity", "level", "CollegeFinancialState", "InstitutionType",
        "category", "COUNTRY", "Institute_Address_Line_1", "ZIP_COMPLETE", "line1",
        "orgId", "type", "years", "CollegeCampusZipCode", "InstitutionCarnegieId",
        "OsageClassification", "CurrentClassification", "status", "CollegeCampusAddress1",
        "CollegeCampusState", "TitleIV", "CITY", "Institute_Address_Line_2", "size",
        "CollegeCampusAddress2", "CollegeFinancialZipCode", "Ipeds", "Basic2015", "Code",
        "InstitutionControlTypeDescription", "ceeb", "state", "charter",
        "Basic2015Description", "IsAccredited", "postal", "label",
    ],
    "ein": ["ein"],
    "organizationein": ["ein"],
    "calculatedValue": ["calculatedValue"],
    "likertscale": ["label"],
    "number": ["number"],
    "email": ["email"],
    "primaryemail": ["email"],
    "gender": ["label"],
    "name": ["middle", "prefix", "suffix", "first", "last"],
    "individualname": ["middle", "prefix", "suffix", "first", "last"],
    "organizationlegalname": ["text"],
    "contacts": ["email", "first", "title", "last", "phone"],
    "organizationcontacts": ["email", "first", "title", "last", "phone"],
    "singleselect": ["label"],
    "text": ["text"],
    "numericslider": ["number"],
}


def _convert_sub_field_id(sub_field_id: str) -> str:
    mapping = {
        "startDate": "Start Date",
        "date": "Date",
        "endDate": "End Date",
        "line1": "Line 1",
        "line2": "Line 2",
        "city": "City",
        "state": "State",
        "country": "Country",
        "postal": "Postal",
        "prefix": "Prefix",
        "first": "First",
        "middle": "Middle",
        "last": "Last",
        "suffix": "Suffix",
        "email": "Email",
        "phone": "Phone",
        "title": "Title",
        # These collapse to no readable suffix.
        "label": "",
        "text": "",
        "number": "",
        "ein": "",
        "calculatedValue": "",
    }
    return mapping.get(sub_field_id, sub_field_id)


def _base_field_display_name(f: IntakeFieldInput) -> str:
    """Display label from Snowflake; full ``field_id`` is appended separately."""
    t = (f.field_label or "").strip()
    return t or f"Field {f.field_id[:8]}"


def _is_record(x: Any) -> bool:
    return isinstance(x, dict)


def _ensure_data_panel_groups(
    model: Dict[str, Any], program_group_names: Iterable[str]
) -> None:
    """Idempotently register program-name groups under a DATA_PANEL
    ``column_groups`` entry, forcing ``properties.status`` to ``ENABLE``.
    """
    groups: List[Dict[str, Any]] = (
        [g for g in model["column_groups"] if _is_record(g)]
        if isinstance(model.get("column_groups"), list)
        else []
    )

    data_panel: Optional[Dict[str, Any]] = next(
        (g for g in groups if g.get("type") == "DATA_PANEL"), None
    )
    if data_panel is None:
        data_panel = {
            "type": "DATA_PANEL",
            "properties": {"status": "ENABLE", "default_sort": "DISABLE"},
            "column_group_info": [],
        }
        groups.append(data_panel)
    if _is_record(data_panel.get("properties")):
        data_panel["properties"]["status"] = "ENABLE"
    else:
        data_panel["properties"] = {"status": "ENABLE"}

    info: List[Dict[str, Any]] = (
        [g for g in data_panel["column_group_info"] if _is_record(g)]
        if isinstance(data_panel.get("column_group_info"), list)
        else []
    )

    if not any(g.get("name") == "Profile Responses" for g in info):
        info.append({"name": "Profile Responses", "include_ungrouped_columns": False})

    for group_name in program_group_names:
        if (
            not any(g.get("name") == group_name for g in info)
            and group_name != "Profile Responses"
        ):
            info.append({"name": group_name, "include_ungrouped_columns": False})

    data_panel["column_group_info"] = info
    model["column_groups"] = groups


def _get_column_type(field_type: str) -> str:
    """NOTE (faithful port): the TS switch lower-cases the type but compares
    against ``"calculatedValue"``, which never matched -- calculatedValue fields
    are ATTRIBUTE, not MEASURE. Preserved here.
    """
    t = (field_type or "").strip().lower()
    if t == "number":
        return "MEASURE"
    if t == "calculatedValue":  # never matches (t is lower-cased) -- see note
        return "MEASURE"
    return "ATTRIBUTE"


def merge_intake_formulas_into_model_tml(
    doc: Dict[str, Any],
    fields: List[IntakeFieldInput],
    logical_table: str,
) -> Dict[str, Any]:
    out = copy.deepcopy(doc)
    model = out.get("model")
    if not _is_record(model):
        raise ValueError("TML document must have a model object")

    # TS used localeCompare; for the ASCII field ids in use this matches a plain
    # code-point sort.
    ordered = sorted(fields, key=lambda f: f.field_id)

    new_formulas: List[Dict[str, Any]] = []
    new_columns: List[Dict[str, Any]] = []
    # Insertion-ordered de-dup (JS Set iterates in insertion order).
    program_group_names: "dict[str, None]" = {}
    missing_field_types: "dict[str, None]" = {}

    for f in ordered:
        field_type = f.field_type
        if not field_type:
            continue  # no type from Snowflake -> nothing to map.

        sub_field_ids = _FIELD_SUBFIELD_MAPPINGS.get(field_type)
        if sub_field_ids is None:
            missing_field_types[field_type] = None
            continue
        for sub_field_id in sub_field_ids:
            fid = f"{f.field_id}_{sub_field_id}"
            unique_name = f"{_base_field_display_name(f)}: {sub_field_id} ({f.field_id})"
            expr = field_type_to_model_formula_expr(
                field_type, f.field_id, sub_field_id, logical_table
            )
            new_formulas.append({"id": fid, "name": unique_name, "expr": expr})

            column_type = _get_column_type(field_type)

            readable = _convert_sub_field_id(sub_field_id)
            readable_sub_field = "" if len(readable) == 0 else ": " + readable
            unique_column_name = f"{_base_field_display_name(f)}{readable_sub_field} ({f.field_id})"
            column: Dict[str, Any] = {
                "name": unique_column_name,
                "formula_id": fid,
                "properties": {
                    "column_type": column_type,
                    "index_type": "DONT_INDEX",
                },
            }
            program_name = (
                f.group_name.strip().replace(":", "-") if f.group_name else None
            )
            if program_name:
                column["data_panel_column_groups"] = {program_name: None}
                program_group_names[program_name] = None
            new_columns.append(column)

    if missing_field_types:
        print(
            f"Missing {logical_table} Field Types: {','.join(missing_field_types.keys())}"
        )

    existing_formulas = model["formulas"] if isinstance(model.get("formulas"), list) else []
    model["formulas"] = [*existing_formulas, *new_formulas]

    existing_cols = (
        [c for c in model["columns"] if _is_record(c)]
        if isinstance(model.get("columns"), list)
        else []
    )
    model["columns"] = [*existing_cols, *new_columns]

    _ensure_data_panel_groups(model, program_group_names.keys())

    return out

### ThoughtSpot REST — HTTP helper + auth token
`post_json` is a thin `requests` wrapper (never throws on non-2xx). `mint_full_bearer_token` mints a bearer token scoped to a ThoughtSpot org.

In [ ]:
# --- _http ---
"""Thin ``requests`` wrapper standing in for axios.

The TypeScript code called axios with ``validateStatus: () => true`` (never throw
on non-2xx) and then inspected ``res.status`` / ``res.data`` itself. ``requests``
already behaves that way -- it only raises via ``raise_for_status()``, which we
never call -- so :func:`post_json` returns ``(status, data)`` just like the TS
callers expect.
"""


from typing import Any, Dict, Optional, Tuple

import requests


def auth_headers(token: str) -> Dict[str, str]:
    return {
        "Authorization": f"Bearer {token}",
        "Accept": "application/json",
        "Content-Type": "application/json",
    }


def _timeout(timeout_s: Optional[float]):
    # axios/requests: 0 (or None) => no timeout. requests uses None for that.
    if timeout_s is None or timeout_s <= 0:
        return None
    return timeout_s


def post_json(
    url: str,
    body: Any,
    headers: Optional[Dict[str, str]] = None,
    timeout_s: Optional[float] = 0,
) -> Tuple[int, Any]:
    """POST JSON and return ``(status_code, parsed_data)``.

    ``parsed_data`` is the decoded JSON when possible, otherwise the raw text.
    """
    res = requests.post(url, json=body, headers=headers, timeout=_timeout(timeout_s))
    try:
        data: Any = res.json()
    except ValueError:
        data = res.text
    return res.status_code, data

# --- mint_token ---
"""Mint a full bearer token (port of ``src/thoughtspot/mintToken.ts``)."""


import json
from typing import Any, Optional


_VALIDITY_SEC = 86_400


def _truncate(data: Any, n: int = 500) -> str:
    try:
        return json.dumps(data)[:n]
    except (TypeError, ValueError):
        return str(data)[:n]


def _extract_token(data: Any) -> Optional[str]:
    if isinstance(data, str) and len(data) > 0:
        return data
    if not isinstance(data, dict):
        return None
    direct = data.get("token") or data.get("bearer_token") or data.get("access_token")
    if isinstance(direct, str) and len(direct) > 0:
        return direct
    inner = data.get("data")
    if isinstance(inner, dict):
        t = inner.get("token") or inner.get("bearer_token") or inner.get("access_token")
        if isinstance(t, str) and len(t) > 0:
            return t
    return None


def mint_full_bearer_token(host: str, username: str, secret_key: str, org_id: int) -> str:
    """POST /api/rest/2.0/auth/token/full.

    Mints a full bearer token scoped to ``org_id`` (``0`` = primary org).
    ``auto_create`` is fixed to False; validity fixed to 24h.
    """
    url = f"{host}/api/rest/2.0/auth/token/full"
    body = {
        "username": username,
        "org_id": org_id,
        "validity_time_in_sec": _VALIDITY_SEC,
        "auto_create": False,
        "secret_key": secret_key,
    }
    status, data = post_json(
        url, body, headers={"Accept": "application/json", "Content-Type": "application/json"}
    )
    if status < 200 or status >= 300:
        raise RuntimeError(
            f"auth/token/full failed for org_id={org_id}: HTTP {status} {_truncate(data)}"
        )
    token = _extract_token(data)
    if not token:
        raise RuntimeError(
            f"auth/token/full returned no token string for org_id={org_id}: {_truncate(data)}"
        )
    return token

### ThoughtSpot REST — org discovery
`list_orgs` / `find_org_id_by_name` map a Submittable org id to its ThoughtSpot org (`Next-<id>`).

In [ ]:
# --- orgs ---
"""Org discovery (port of ``src/thoughtspot/orgs.ts``)."""


import json
import math
from dataclasses import dataclass
from typing import Any, Dict, List, Optional



@dataclass
class ThoughtSpotOrg:
    org_id: int
    name: str


def _truncate(data: Any, n: int = 500) -> str:
    try:
        return json.dumps(data)[:n]
    except (TypeError, ValueError):
        return str(data)[:n]


def _normalize_org_rows(data: Any) -> List[Any]:
    if isinstance(data, list):
        return data
    if not isinstance(data, dict):
        return []
    for key in ("orgs", "results", "records", "data"):
        v = data.get(key)
        if isinstance(v, list):
            return v
    return []


def _row_to_org(row: Any) -> Optional[ThoughtSpotOrg]:
    if not isinstance(row, dict):
        return None
    raw_id = row.get("id")
    if raw_id is None:
        raw_id = row.get("org_id")
    if raw_id is None:
        raw_id = row.get("orgId")
    raw_name = row.get("name") or row.get("org_name") or row.get("orgName")
    if raw_id is None:
        return None
    try:
        org_num = float(raw_id) if not isinstance(raw_id, (int, float)) else float(raw_id)
    except (TypeError, ValueError):
        return None
    if not math.isfinite(org_num):
        return None
    org_id = int(org_num)
    name = raw_name.strip() if isinstance(raw_name, str) else ""
    if not name:
        return None
    return ThoughtSpotOrg(org_id=org_id, name=name)


def _post_orgs_search(host: str, bearer_token: str, body: Dict[str, Any]):
    url = f"{host}/api/rest/2.0/orgs/search"
    return post_json(url, body, headers=auth_headers(bearer_token))


def list_orgs(host: str, bearer_token: str) -> List[ThoughtSpotOrg]:
    """List every org visible to ``bearer_token`` (requires an org_id=0 token)."""
    status, data = _post_orgs_search(host, bearer_token, {"record_offset": 0, "record_size": -1})
    if status < 200 or status >= 300:
        raise RuntimeError(f"orgs/search failed: HTTP {status} {_truncate(data)}")
    rows = _normalize_org_rows(data)
    seen = set()
    orgs: List[ThoughtSpotOrg] = []
    for row in rows:
        o = _row_to_org(row)
        if not o or o.org_id in seen:
            continue
        seen.add(o.org_id)
        orgs.append(o)
    return orgs


def find_org_id_by_name(host: str, bearer_token: str, name: str) -> Optional[int]:
    """Resolve a TS org id from an exact-match name. None if none; raises if many."""
    name = (name or "").strip()
    if not name:
        return None

    status, data = _post_orgs_search(
        host, bearer_token, {"org_identifier": name, "record_offset": 0, "record_size": 50}
    )
    if status < 200 or status >= 300:
        raise RuntimeError(f'orgs/search failed for "{name}": HTTP {status} {_truncate(data)}')

    rows = _normalize_org_rows(data)
    matches: List[ThoughtSpotOrg] = []
    for row in rows:
        o = _row_to_org(row)
        if o and o.name == name:
            matches.append(o)
    if len(matches) == 0:
        return None
    if len(matches) > 1:
        ids = ", ".join(str(m.org_id) for m in matches)
        raise RuntimeError(f'Multiple ThoughtSpot orgs named "{name}" (ids: {ids}). Cannot pick one.')
    return matches[0].org_id

### ThoughtSpot REST — export the template model
Search a model by name and export its TML YAML (tries WORKSHEET then MODEL). `find_existing_model_by_name_if_unique` resolves create-vs-update by name + obj_id.

In [ ]:
# --- fetch_model_tml ---
"""Search for a model by name and export its TML YAML.

Port of ``src/thoughtspot/fetchModelTmlByName.ts``. Tries LOGICAL_TABLE+WORKSHEET
first, then native MODEL. Throws on ambiguous matches or no match.
"""


import json
import os
from dataclasses import dataclass
from typing import Any, Dict, List, Optional


# Search strategies: worksheets (legacy "model" in UI) then native MODEL.
_SEARCH_STRATEGIES: List[Dict[str, Any]] = [
    {"type": "LOGICAL_TABLE", "subtypes": ["WORKSHEET"]},
    {"type": "MODEL"},
]


def _truncate(data: Any, n: int = 500) -> str:
    try:
        return json.dumps(data)[:n]
    except (TypeError, ValueError):
        return str(data)[:n]


def _is_record(x: Any) -> bool:
    return isinstance(x, dict)


def _normalize_search_rows(data: Any) -> List[Any]:
    if isinstance(data, list):
        return data
    if not isinstance(data, dict):
        return []
    if isinstance(data.get("results"), list):
        return data["results"]
    if isinstance(data.get("metadata"), list):
        return data["metadata"]
    if isinstance(data.get("records"), list):
        return data["records"]
    return []


def _hit_display_name(hit: Dict[str, Any]) -> str:
    n = hit.get("metadata_name")
    if isinstance(n, str) and n.strip():
        return n.strip()
    hdr = hit.get("metadata_header")
    if _is_record(hdr):
        hn = hdr.get("name")
        if isinstance(hn, str) and hn.strip():
            return hn.strip()
    return ""


def _hit_guid(hit: Dict[str, Any]) -> Optional[str]:
    mid = hit.get("metadata_id")
    if isinstance(mid, str) and len(mid) > 0:
        return mid
    hdr = hit.get("metadata_header")
    if _is_record(hdr):
        _id = hdr.get("id_guid") or hdr.get("id") or hdr.get("guid")
        if isinstance(_id, str) and len(_id) > 0:
            return _id
    return None


def _hit_metadata_type(hit: Dict[str, Any]) -> Optional[str]:
    t = hit.get("metadata_type")
    return t if isinstance(t, str) and len(t) > 0 else None


def _hit_obj_id(hit: Dict[str, Any]) -> Optional[str]:
    direct = hit.get("metadata_obj_id") or hit.get("obj_id")
    if isinstance(direct, str) and len(direct) > 0:
        return direct
    hdr = hit.get("metadata_header")
    if _is_record(hdr):
        obj_id = hdr.get("obj_id")
        if isinstance(obj_id, str) and len(obj_id) > 0:
            return obj_id
    return None


def _filter_hits_by_exact_name(rows: List[Any], wanted_name: str) -> List[Dict[str, Any]]:
    want = wanted_name.strip().lower()
    out: List[Dict[str, Any]] = []
    for row in rows:
        if not _is_record(row):
            continue
        if _hit_display_name(row).lower() == want:
            out.append(row)
    return out


def _post_search(host: str, bearer_token: str, strategy: Dict[str, Any], name: str):
    url = f"{host}/api/rest/2.0/metadata/search"
    meta: Dict[str, Any] = {"type": strategy["type"], "identifier": name.strip()}
    if strategy.get("subtypes"):
        meta["subtypes"] = strategy["subtypes"]
    return post_json(
        url,
        {"metadata": [meta], "record_size": 50, "record_offset": 0},
        headers=auth_headers(bearer_token),
    )


@dataclass
class ResolvedHit:
    guid: str
    metadata_type: str
    display_name: str
    obj_id: Optional[str]


def _resolve_hit(hit: Dict[str, Any], fallback_name: str, fallback_type: str) -> Optional[ResolvedHit]:
    guid = _hit_guid(hit)
    if not guid:
        return None
    return ResolvedHit(
        guid=guid,
        metadata_type=_hit_metadata_type(hit) or fallback_type,
        display_name=_hit_display_name(hit) or fallback_name,
        obj_id=_hit_obj_id(hit),
    )


def _search_model_by_name_outcome(host: str, bearer_token: str, model_name: str) -> Dict[str, Any]:
    """Returns {"kind": "one"|"none"|"many", ...} mirroring the TS union."""
    name = (model_name or "").strip()
    if not name:
        return {"kind": "none", "last_http_status": 0}

    last_status = 0
    for strategy in _SEARCH_STRATEGIES:
        status, data = _post_search(host, bearer_token, strategy, name)
        last_status = status
        if status < 200 or status >= 300:
            continue

        rows = _normalize_search_rows(data)
        matched = _filter_hits_by_exact_name(rows, name)

        if len(matched) == 1:
            resolved = _resolve_hit(matched[0], name, strategy["type"])
            if not resolved:
                return {"kind": "none", "last_http_status": status}
            return {"kind": "one", "hit": resolved}
        if len(matched) > 1:
            hits: List[ResolvedHit] = []
            for m in matched:
                r = _resolve_hit(m, name, strategy["type"])
                if r:
                    hits.append(r)
            return {"kind": "many", "hits": hits}

    return {"kind": "none", "last_http_status": last_status}


def _describe_many_hits(hits: List[ResolvedHit]) -> str:
    parts = []
    for h in hits:
        obj = f" obj_id={h.obj_id}" if h.obj_id else ""
        parts.append(f"{h.display_name} (guid={h.guid}{obj})")
    return "; ".join(parts)


def find_existing_model_by_name_if_unique(
    host: str,
    bearer_token: str,
    model_name: str,
    expected_obj_id: Optional[str] = None,
) -> Optional[Dict[str, str]]:
    """If exactly one object matches ``model_name``, return ``{guid, metadata_type}``;
    None if no match. When several share the name, ``expected_obj_id``
    disambiguates (the one we previously imported under our stable obj_id);
    otherwise raise so the caller can clean up the duplicate.
    """
    out = _search_model_by_name_outcome(host, bearer_token, model_name)
    if out["kind"] == "one":
        hit: ResolvedHit = out["hit"]
        return {"guid": hit.guid, "metadata_type": hit.metadata_type}
    if out["kind"] == "many":
        name = (model_name or "").strip()
        hits: List[ResolvedHit] = out["hits"]
        expected = expected_obj_id
        if expected:
            ours = [h for h in hits if h.obj_id == expected]
            if len(ours) == 1:
                return {"guid": ours[0].guid, "metadata_type": ours[0].metadata_type}
            if len(ours) == 0:
                raise RuntimeError(
                    f'Multiple ThoughtSpot objects named "{name}" ({_describe_many_hits(hits)}), '
                    f'none with obj_id "{expected}". A name-clashing object exists in this org; '
                    "rename or delete the conflict in ThoughtSpot before re-running."
                )
            raise RuntimeError(
                f'Multiple ThoughtSpot objects named "{name}" share obj_id "{expected}" '
                f"({_describe_many_hits(ours)}). Delete the duplicates in ThoughtSpot before re-running."
            )
        raise RuntimeError(
            f'Multiple ThoughtSpot objects named "{name}" ({_describe_many_hits(hits)}). '
            "Cannot pick one to upsert."
        )
    return None


def _looks_like_tml_yaml(s: str) -> bool:
    t = s.strip()
    if len(t) < 40:
        return False
    normalized = t.replace("\\n", "\n") if ("\\n" in t and "\n" not in t) else t
    has_break = "\n" in normalized or "\\n" in normalized
    has_tml_shape = (
        "guid:" in normalized
        or "model:" in normalized
        or "table:" in normalized
        or normalized.lstrip().startswith("---")
    )
    if not has_tml_shape:
        return False
    if has_break:
        return True
    return "model:" in normalized and "guid:" in normalized and len(normalized) > 120


def _normalize_exported_yaml_string(s: str) -> str:
    t = s.strip()
    if t.startswith('"') and t.endswith('"'):
        try:
            parsed = json.loads(t)
            if isinstance(parsed, str):
                t = parsed
        except (ValueError, TypeError):
            pass  # keep t
    if "\\n" in t and "\n" not in t:
        t = t.replace("\\n", "\n").replace('\\"', '"')
    return t


def _find_tml_yaml_deep(value: Any, depth: int = 0) -> Optional[str]:
    """Depth-first search for a TML YAML string (response shapes vary by version)."""
    if depth > 14:
        return None

    if isinstance(value, str):
        norm = _normalize_exported_yaml_string(value)
        return norm if _looks_like_tml_yaml(norm) else None

    if isinstance(value, list):
        for item in value:
            found = _find_tml_yaml_deep(item, depth + 1)
            if found:
                return found
        return None

    if not _is_record(value):
        return None

    priority_keys = (
        "metadata_tmls",
        "metadata_tml",
        "edoc",
        "tml",
        "content",
        "yaml",
        "document",
        "metadata_tml_string",
        "tml_string",
    )
    for key in priority_keys:
        sub = value.get(key)
        if isinstance(sub, str):
            norm = _normalize_exported_yaml_string(sub)
            if _looks_like_tml_yaml(norm):
                return norm
        if isinstance(sub, list):
            found = _find_tml_yaml_deep(sub, depth + 1)
            if found:
                return found

    for v in value.values():
        found = _find_tml_yaml_deep(v, depth + 1)
        if found:
            return found
    return None


def _unwrap_export_payload(data: Any, depth: int = 0) -> Any:
    if depth > 6 or not _is_record(data):
        return data
    inner = data.get("data")
    if inner is None:
        inner = data.get("response")
    if inner is None:
        inner = data.get("result")
    if inner is None:
        inner = data.get("body")
    if inner is not None and inner is not data:
        return _unwrap_export_payload(inner, depth + 1)
    return data


def _export_tml_yaml(host: str, bearer_token: str, guid: str, metadata_type: str) -> str:
    url = f"{host}/api/rest/2.0/metadata/tml/export"
    body: Dict[str, Any] = {
        "metadata": [{"type": metadata_type, "identifier": guid}],
        "edoc_format": "YAML",
        "export_associated": False,
        # Include each referenced table's GUID as `fqn`. Without this, import
        # fails when the target org has same-named tables with different GUIDs.
        "export_fqn": True,
    }
    if metadata_type in ("MODEL", "LOGICAL_TABLE"):
        body["export_schema_version"] = "V2"

    status, data = post_json(url, body, headers=auth_headers(bearer_token))
    if status < 200 or status >= 300:
        raise RuntimeError(
            f"ThoughtSpot metadata/tml/export failed: HTTP {status} {_truncate(data)}"
        )

    yaml = _find_tml_yaml_deep(_unwrap_export_payload(data))
    if not yaml:
        hint = ""
        if isinstance(data, dict):
            hint = f" Top-level keys: {', '.join(data.keys())}."
        detail = (
            f" Body: {_truncate(data, 4000)}"
            if os.environ.get("DEBUG") == "1"
            else " Set DEBUG=1 to log the response body."
        )
        raise RuntimeError(
            f"metadata/tml/export returned no YAML string in a recognized shape.{hint}{detail}"
        )
    return yaml


def fetch_model_tml_yaml_by_name(host: str, bearer_token: str, model_name: str) -> str:
    """Search ThoughtSpot for a model by display name, then export it as YAML."""
    name = (model_name or "").strip()
    if not name:
        raise ValueError("model_name must be non-empty")

    out = _search_model_by_name_outcome(host, bearer_token, name)
    if out["kind"] == "many":
        raise RuntimeError(
            f'Multiple ThoughtSpot objects named "{name}": {_describe_many_hits(out["hits"])}. '
            "Rename one in ThoughtSpot."
        )
    if out["kind"] == "none":
        last = out.get("last_http_status", 0)
        hint = (
            f"Last HTTP status {last}."
            if last >= 400
            else "No exact name match after trying LOGICAL_TABLE/WORKSHEET and MODEL."
        )
        raise RuntimeError(
            f'ThoughtSpot metadata/search: no unique object named "{name}". {hint} '
            "Verify type/subtype on your cluster."
        )
    hit: ResolvedHit = out["hit"]
    return _export_tml_yaml(host, bearer_token, hit.guid, hit.metadata_type)

### ThoughtSpot REST — async TML import
`import_table_tml` submits the merged TML, polls to completion, and interprets the response (including nested error codes). Plus helpers to read the GUID and success status.

In [ ]:
# --- import_table_tml ---
"""Async TML import + response interpretation.

Port of ``src/thoughtspot/importTableTml.ts``. Kicks off
``metadata/tml/async/import`` (returns a task_id), polls
``metadata/tml/async/status`` until terminal, and returns the inner
``import_response`` envelope so :func:`is_thoughtspot_import_response_ok` and
:func:`extract_guid_from_import_response` work against it unchanged.
"""


import json
import os
import re
import time
from dataclasses import dataclass
from typing import Any, Callable, Dict, List, Optional


_TERMINAL_TASK_STATUSES = {"COMPLETED", "FAILED", "ERROR", "CANCELLED"}
_FAILED_TASK_STATUSES = {"FAILED", "ERROR", "CANCELLED"}

# Standard + ThoughtSpot-style hyphenated 32-hex ids (not strict RFC 4122 variant).
_UUID_RE = re.compile(
    r"^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$", re.IGNORECASE
)

_TERMINAL_IMPORT_STATUSES = {"ERROR", "FAILED", "FAILURE"}


def _is_record(x: Any) -> bool:
    return isinstance(x, dict)


def _is_uuid(s: Any) -> bool:
    return isinstance(s, str) and bool(_UUID_RE.match(s))


def _parse_poll_env(name: str, fallback: int) -> int:
    raw = os.environ.get(name)
    if raw is None or raw.strip() == "":
        return fallback
    try:
        n = int(raw, 10)
    except ValueError:
        return fallback
    if n < 0:
        return fallback
    return n


def _guid_from_tml_import_entry(entry: Any) -> Optional[str]:
    if not _is_record(entry):
        return None
    envelope = entry.get("response")
    if envelope is None:
        envelope = entry.get("import_response")
    if not _is_record(envelope):
        return None
    header = envelope.get("header")
    if not _is_record(header):
        return None
    _id = header.get("id_guid") or header.get("guid")
    if _is_uuid(_id):
        return _id
    return None


def extract_guid_from_import_response(data: Any) -> Optional[str]:
    if data is None:
        return None
    if _is_uuid(data):
        return data
    if not isinstance(data, (dict, list)):
        return None

    if isinstance(data, list):
        for item in data:
            g = _guid_from_tml_import_entry(item) or extract_guid_from_import_response(item)
            if g:
                return g
        return None

    o: Dict[str, Any] = data

    from_batch_entry = _guid_from_tml_import_entry(o)
    if from_batch_entry:
        return from_batch_entry

    wrapped = o.get("data")
    if wrapped is None:
        wrapped = o.get("body")
    if wrapped is not None and wrapped is not o:
        g = extract_guid_from_import_response(wrapped)
        if g:
            return g

    direct = o.get("guid") or o.get("id") or o.get("id_guid")
    if _is_uuid(direct):
        return direct

    header = o.get("header")
    if header is None:
        header = o.get("metadata_header")
    if isinstance(header, dict):
        hid = header.get("id_guid") or header.get("id") or header.get("guid")
        if _is_uuid(hid):
            return hid

    info = o.get("info")
    if isinstance(info, dict):
        iid = info.get("id_guid") or info.get("id") or info.get("guid")
        if _is_uuid(iid):
            return iid

    candidates = [
        o.get("table"),
        o.get("model"),
        o.get("metadata_object"),
        o.get("metadata_objects"),
        o.get("object"),
        o.get("objects"),
        o.get("import_response"),
        o.get("response"),
        o.get("result"),
    ]
    for c in candidates:
        g = extract_guid_from_import_response(c)
        if g:
            return g

    for v in o.values():
        if _is_uuid(v):
            return v
        if isinstance(v, dict):
            ng = v.get("id_guid") or v.get("guid") or v.get("id")
            if _is_uuid(ng):
                return ng

    return None


def _read_object_status_code(obj: Dict[str, Any]) -> Optional[str]:
    direct = obj.get("status_code")
    if direct is None:
        direct = obj.get("statusCode")
    if isinstance(direct, str):
        return direct
    st = obj.get("status")
    if _is_record(st):
        nested = st.get("status_code")
        if nested is None:
            nested = st.get("statusCode")
        if isinstance(nested, str):
            return nested
    return None


def _push_status_from_object(obj: Any, out: List[str]) -> None:
    if not _is_record(obj):
        return
    c = _read_object_status_code(obj)
    if c:
        out.append(c)


def _is_nonzero_number(x: Any) -> bool:
    # TS used `typeof x === "number" && x !== 0`, which accepts floats. A JSON
    # `error_code` of 20001.0 must count -- `isinstance(int)` alone would miss it
    # and wrongly treat a failed import as OK. Exclude bool (an int subclass).
    return isinstance(x, (int, float)) and not isinstance(x, bool) and x != 0


def _read_object_error_code(obj: Dict[str, Any]) -> Optional[int]:
    direct = obj.get("error_code")
    if _is_nonzero_number(direct):
        return direct
    st = obj.get("status")
    if _is_record(st):
        nested = st.get("error_code")
        if _is_nonzero_number(nested):
            return nested
    return None


def extract_import_error_code(data: Any) -> Optional[int]:
    """First non-zero ``error_code`` in a TML import response. ThoughtSpot returns
    HTTP 200 for partial failures and hides the real code at
    ``object[*].response.status.error_code``.
    """
    if not _is_record(data):
        return None

    for key in ("import_response", "response", "result"):
        sub = data.get(key)
        if _is_record(sub):
            c = _read_object_error_code(sub)
            if c is not None:
                return c

    for key in ("metadata_objects", "objects", "object"):
        arr = data.get(key)
        if isinstance(arr, list):
            for item in arr:
                if not _is_record(item):
                    continue
                from_response = (
                    _read_object_error_code(item["response"])
                    if _is_record(item.get("response"))
                    else None
                )
                c = from_response if from_response is not None else _read_object_error_code(item)
                if c is not None:
                    return c

    return _read_object_error_code(data)


def is_thoughtspot_import_response_ok(status: int, data: Any) -> bool:
    """Whether a TML import HTTP 2xx response represents success.

    Avoids scanning the entire JSON tree: nested objects may carry their own
    ``status_code`` values that are not overall import failures.
    """
    if status < 200 or status >= 300:
        return False
    if data is None:
        return True
    if not isinstance(data, dict):
        return True

    root: Dict[str, Any] = data

    err = root.get("error")
    if err is not None:
        if isinstance(err, str) and err.strip() != "":
            return False
        if _is_record(err):
            return False

    # HTTP 200 can still carry a nested non-zero error_code.
    if extract_import_error_code(root) is not None:
        return False

    codes: List[str] = []
    _push_status_from_object(root, codes)

    for key in ("import_response", "response", "result"):
        _push_status_from_object(root.get(key), codes)

    for key in ("metadata_objects", "objects", "object"):
        arr = root.get(key)
        if isinstance(arr, list):
            for item in arr[:20]:
                _push_status_from_object(item, codes)
                if _is_record(item) and _is_record(item.get("response")):
                    _push_status_from_object(item["response"], codes)

    for key in ("metadata_object", "model", "table"):
        _push_status_from_object(root.get(key), codes)

    if len(codes) == 0:
        return True
    return not any(c.upper() in _TERMINAL_IMPORT_STATUSES for c in codes)


@dataclass
class TableTmlImportResult:
    status: int
    data: Any
    guid: Optional[str] = None


def import_table_tml(
    host: str,
    bearer_token: str,
    tml_yaml: str,
    create_new: bool,
    import_policy: str,
    timeout_s: float = 0,
    on_log: Optional[Callable[[str], None]] = None,
) -> TableTmlImportResult:
    headers = {
        "Authorization": f"Bearer {bearer_token}",
        "Accept": "application/json",
        "Content-Type": "application/json",
    }
    timeout = timeout_s if timeout_s and timeout_s > 0 else 0

    import_url = f"{host}/api/rest/2.0/metadata/tml/async/import"
    submit_status, submit_data = post_json(
        import_url,
        {
            "metadata_tmls": [tml_yaml],
            "create_new": create_new,
            "import_policy": import_policy,
        },
        headers=headers,
        timeout_s=timeout,
    )

    if submit_status < 200 or submit_status >= 300:
        return TableTmlImportResult(status=submit_status, data=submit_data)

    submit_body = submit_data if _is_record(submit_data) else None
    task_id = submit_body.get("task_id") if submit_body else None
    if not isinstance(task_id, str) or len(task_id) == 0:
        raise RuntimeError(
            f"tml/async/import returned no task_id: {json.dumps(submit_data)[:500]}"
        )
    log = on_log or (lambda msg: print(msg))
    log(f"tml/async/import accepted: task_id={task_id}")

    poll_interval_s = _parse_poll_env("TML_POLL_INTERVAL_MS", 2000) / 1000.0
    poll_timeout_ms = _parse_poll_env("TML_POLL_TIMEOUT_MS", 0)  # 0 = never time out.

    status_url = f"{host}/api/rest/2.0/metadata/tml/async/status"
    deadline = (time.time() + poll_timeout_ms / 1000.0) if poll_timeout_ms > 0 else float("inf")
    entry: Optional[Dict[str, Any]] = None
    last_task_status = ""

    while True:
        st_status, st_data = post_json(
            status_url,
            {"task_ids": [task_id], "include_import_response": True},
            headers=headers,
            timeout_s=timeout,
        )
        if st_status < 200 or st_status >= 300:
            return TableTmlImportResult(status=st_status, data=st_data)

        _list = st_data.get("status_list") if _is_record(st_data) else None
        entry = _list[0] if isinstance(_list, list) and _list and _is_record(_list[0]) else None
        ts = entry.get("task_status") if entry and isinstance(entry.get("task_status"), str) else ""
        last_task_status = ts or ""
        if last_task_status.upper() in _TERMINAL_TASK_STATUSES:
            break

        if time.time() >= deadline:
            raise RuntimeError(
                f'tml/async timed out after {poll_timeout_ms}ms '
                f'(last task_status="{last_task_status}", task_id={task_id})'
            )
        time.sleep(poll_interval_s)

    import_response = entry.get("import_response") if entry else None
    guid = extract_guid_from_import_response(import_response)

    if last_task_status.upper() in _FAILED_TASK_STATUSES:
        return TableTmlImportResult(status=500, data=entry, guid=guid)

    return TableTmlImportResult(status=submit_status, data=import_response, guid=guid)

### Config
`config_from_mapping(dict)` builds the typed config (ThoughtSpot host/creds, import policy, template name). In the notebook we pass a plain dict, so notebook env vars never leak in.

In [ ]:
# --- config ---
"""Application config (port of ``src/config.ts``).

Two loading paths:

* :func:`load_config` reads the same environment variables the TypeScript CLI
  used (via ``.env`` when ``python-dotenv`` is installed). Use this for local
  runs and CI.
* :func:`config_from_mapping` builds the same config from a plain dict, which is
  the convenient path inside a Snowflake Notebook: read your secrets from
  Snowflake Secrets / notebook widgets and hand them in directly.

Inside a Snowflake Notebook the Snowflake connection comes from the active
Snowpark session, so ``SNOWFLAKE_PRIVATE_KEY`` (and the other Snowflake
connection params) are optional there -- see ``snowflake_fetch.py``.
"""


from dataclasses import dataclass
from typing import Mapping, Optional

try:  # optional: load .env for local/CI runs. No-op inside Snowflake Notebooks.
    from pathlib import Path

    from dotenv import load_dotenv

    # Deterministically load the package-local .env (ts_sweeper/.env) first, so
    # it works regardless of the current working directory. `override=False`
    # (the default) means real environment variables still win. Then fall back
    # to a .env discovered in the CWD / its ancestors, if any.
    _pkg_env = Path(__file__).with_name(".env")
    if _pkg_env.exists():
        load_dotenv(_pkg_env)
    load_dotenv()
except Exception:  # pragma: no cover - dotenv is optional
    pass

import os

ThoughtSpotImportPolicy = str  # "PARTIAL" | "ALL_OR_NONE" | "VALIDATE_ONLY" | "PARTIAL_OBJECT"

_ALLOWED_IMPORT_POLICIES = (
    "PARTIAL",
    "ALL_OR_NONE",
    "VALIDATE_ONLY",
    "PARTIAL_OBJECT",
)


@dataclass
class SnowflakeConfig:
    account: Optional[str]
    username: Optional[str]
    warehouse: Optional[str]
    database: str
    schema: str
    role: Optional[str]
    # PEM private key (key-pair JWT). Optional: unused when a Snowpark session is
    # supplied (the notebook case).
    private_key: Optional[str]


@dataclass
class ThoughtSpotConfig:
    host: str
    # Optional pre-minted bearer token. When set, used for both export and import
    # phases (skips the auth/token/full mint flow).
    bearer_token: Optional[str]
    # Required when `bearer_token` is unset.
    username: Optional[str]
    # Required when `bearer_token` is unset.
    secret_key: Optional[str]
    import_policy: ThoughtSpotImportPolicy


@dataclass
class TmlConfig:
    # ThoughtSpot template model name for metadata/export (same as `TML_MODEL_NAME`).
    model_name: Optional[str]
    # Per-HTTP timeout (seconds) for the TML import POST and each status poll. 0 = no timeout.
    import_timeout_s: float
    # When true, write a support bundle on successful imports too (not just failures).
    support_bundle_on_success: bool


@dataclass
class AppConfig:
    snowflake: SnowflakeConfig
    thoughtspot: ThoughtSpotConfig
    tml: TmlConfig


# ---------------------------------------------------------------------------
# env helpers (mirror the TS require/optional/flag helpers)
# ---------------------------------------------------------------------------


def _get(source: Mapping[str, str], name: str) -> Optional[str]:
    v = source.get(name)
    if v is None:
        return None
    v = v.strip()
    return v or None


def _require(source: Mapping[str, str], name: str) -> str:
    v = _get(source, name)
    if v is None:
        raise ValueError(f"Missing required environment variable: {name}")
    return v


def _parse_non_negative_int(source: Mapping[str, str], name: str, fallback: int) -> int:
    v = _get(source, name)
    if v is None:
        return fallback
    try:
        n = int(v)
    except ValueError:
        raise ValueError(f'Invalid {name}="{v}": expected a non-negative integer.')
    if n < 0:
        raise ValueError(f'Invalid {name}="{v}": expected a non-negative integer.')
    return n


def _flag_true(source: Mapping[str, str], name: str) -> bool:
    v = _get(source, name)
    if not v:
        return False
    return v == "1" or v.lower() == "true" or v.lower() == "yes"


def _load_private_key_pem(source: Mapping[str, str]) -> Optional[str]:
    raw = _get(source, "SNOWFLAKE_PRIVATE_KEY")
    if raw is None:
        return None
    # TS stored the PEM on one line with literal "\n"; expand them back.
    return raw.replace("\\n", "\n")


def _parse_import_policy(raw: Optional[str]) -> ThoughtSpotImportPolicy:
    v = (raw or "PARTIAL").upper()
    if v in _ALLOWED_IMPORT_POLICIES:
        return v
    raise ValueError(
        f'Invalid THOUGHTSPOT_IMPORT_POLICY "{raw}". '
        f"Expected one of: {', '.join(_ALLOWED_IMPORT_POLICIES)}"
    )


def config_from_mapping(source: Mapping[str, str], *, require_snowflake_key: bool = False) -> AppConfig:
    """Build an :class:`AppConfig` from a mapping of the same names the TS env used.

    ``require_snowflake_key`` mirrors the TS behavior (key-pair auth required).
    Leave it False inside a Snowflake Notebook, where the active Snowpark session
    supplies the connection and no private key is needed.
    """
    host = _require(source, "THOUGHTSPOT_HOST").rstrip("/")
    import_policy = _parse_import_policy(source.get("THOUGHTSPOT_IMPORT_POLICY"))
    bearer_token = _get(source, "THOUGHTSPOT_BEARER_TOKEN")
    username = _get(source, "THOUGHTSPOT_USERNAME")
    secret_key = _get(source, "THOUGHTSPOT_SECRET_KEY")

    if not bearer_token and (not username or not secret_key):
        raise ValueError(
            "ThoughtSpot auth missing: set either THOUGHTSPOT_BEARER_TOKEN, or both "
            "THOUGHTSPOT_USERNAME and THOUGHTSPOT_SECRET_KEY."
        )

    private_key = _load_private_key_pem(source)
    if require_snowflake_key and private_key is None:
        raise ValueError("Missing required environment variable: SNOWFLAKE_PRIVATE_KEY")

    return AppConfig(
        snowflake=SnowflakeConfig(
            account=_get(source, "SNOWFLAKE_ACCOUNT"),
            username=_get(source, "SNOWFLAKE_USERNAME"),
            warehouse=_get(source, "SNOWFLAKE_WAREHOUSE"),
            # database/schema default to the Next datamart used by the TS .env.
            database=_get(source, "SNOWFLAKE_DATABASE") or "DATAMART_NEXT",
            schema=_get(source, "SNOWFLAKE_SCHEMA") or "NEXTZEN",
            role=_get(source, "SNOWFLAKE_ROLE"),
            private_key=private_key,
        ),
        thoughtspot=ThoughtSpotConfig(
            host=host,
            bearer_token=bearer_token,
            username=username,
            secret_key=secret_key,
            import_policy=import_policy,
        ),
        tml=TmlConfig(
            model_name=_get(source, "TML_MODEL_NAME"),
            import_timeout_s=float(_parse_non_negative_int(source, "TML_IMPORT_TIMEOUT_MS", 0)) / 1000.0,
            support_bundle_on_success=_flag_true(source, "TML_SUPPORT_BUNDLE_ON_SUCCESS"),
        ),
    )


def load_config(*, require_snowflake_key: bool = False) -> AppConfig:
    """Load config, merging (highest precedence first):

    1. real environment variables (and anything a ``.env`` put there),
    2. the gitignored ``ts_sweeper/settings.py`` ``CONFIG`` dict.

    So env vars still win when set, and inside a Snowflake Notebook -- where
    there are no env vars and no ``.env`` -- ``settings.py`` supplies everything.
    """
    source = dict(os.environ)
    try:
        from settings import CONFIG as _SETTINGS  # gitignored; may be absent

        for k, v in _SETTINGS.items():
            if v is None:
                continue
            source.setdefault(k, str(v))  # fill only what the environment lacks
    except Exception:  # pragma: no cover - settings.py is optional
        pass
    return config_from_mapping(source, require_snowflake_key=require_snowflake_key)

### Snowflake fetch (via the session)
`SnowparkClient` runs SQL on the notebook's active session. `fetch_field_metadata_for_org` runs the field-metadata query (hardcoded to `DATAMART_NEXT.NEXTZEN`) for one org. A key-pair `ConnectorClient` fallback exists for local runs.

In [ ]:
# --- snowflake_fetch ---
"""Snowflake intake-field metadata fetch (port of ``src/snowflake/fetchIntakeFields.ts``).

The TypeScript version always opened a key-pair JWT connection via the Node
Snowflake SDK. In a Snowflake Notebook that's unnecessary: the notebook already
has an authenticated Snowpark session. So this module abstracts the data source
behind a tiny :class:`SnowflakeClient` and picks the right backend:

* ``SnowparkClient`` -- wraps the notebook's active session
  (``get_active_session()``). No private key needed. **Default in notebooks.**
* ``ConnectorClient`` -- key-pair JWT via ``snowflake-connector-python``, for
  local / CI runs (mirrors the original TS behavior).

Call :func:`get_default_client` to auto-select, or construct one explicitly.
"""


import re
from typing import Any, Dict, List, Optional, Protocol, Sequence


_TABLE_NAME_RE = re.compile(r"^[A-Za-z0-9_]+$")


def _assert_safe_table_name(table_name: str) -> None:
    if not _TABLE_NAME_RE.match(table_name):
        raise ValueError(f"Invalid Snowflake table name for field metadata: {table_name}")


# ---------------------------------------------------------------------------
# Data-source abstraction
# ---------------------------------------------------------------------------


class SnowflakeClient(Protocol):
    def query(self, sql: str, binds: Sequence[Any]) -> List[Dict[str, Any]]:
        """Run ``sql`` with positional ``?`` binds and return rows as dicts."""
        ...


class SnowparkClient:
    """Runs queries through a Snowpark session (the Snowflake Notebook case).

    Points the session at the configured database/schema so the field tables
    resolve even when the session's default context is a personal database
    (e.g. ``USER$YOU.PUBLIC``). Best-effort: if the role can't switch context,
    the fully-qualified query itself is the authoritative point of failure.
    """

    def __init__(self, session: Any, database: Optional[str] = None, schema: Optional[str] = None) -> None:
        self._session = session
        for setter_name, value in (("use_database", database), ("use_schema", schema)):
            setter = getattr(session, setter_name, None)
            if setter and value:
                try:
                    setter(value)
                except Exception:
                    pass  # role may lack access; the qualified query will report it

    def query(self, sql: str, binds: Sequence[Any]) -> List[Dict[str, Any]]:
        rows = self._session.sql(sql, params=list(binds)).collect()
        return [row.as_dict() for row in rows]


class ConnectorClient:
    """Key-pair JWT connection via snowflake-connector-python (local / CI)."""

    def __init__(self, cfg: AppConfig) -> None:
        self._cfg = cfg.snowflake

    def query(self, sql: str, binds: Sequence[Any]) -> List[Dict[str, Any]]:
        import snowflake.connector
        from cryptography.hazmat.backends import default_backend
        from cryptography.hazmat.primitives import serialization

        sf = self._cfg
        if not sf.private_key:
            raise ValueError(
                "ConnectorClient requires SNOWFLAKE_PRIVATE_KEY (key-pair auth). "
                "Inside a Snowflake Notebook use SnowparkClient instead."
            )
        pkey = serialization.load_pem_private_key(
            sf.private_key.encode("utf-8"), password=None, backend=default_backend()
        )
        pkb = pkey.private_bytes(
            encoding=serialization.Encoding.DER,
            format=serialization.PrivateFormat.PKCS8,
            encryption_algorithm=serialization.NoEncryption(),
        )
        conn = snowflake.connector.connect(
            account=sf.account,
            user=sf.username,
            warehouse=sf.warehouse,
            database=sf.database,
            schema=sf.schema,
            role=sf.role,
            private_key=pkb,
            paramstyle="qmark",
        )
        try:
            cur = conn.cursor()
            try:
                cur.execute(sql, list(binds))
                columns = [c[0] for c in cur.description]
                return [dict(zip(columns, row)) for row in cur.fetchall()]
            finally:
                cur.close()
        finally:
            conn.close()


def get_default_client(cfg: AppConfig, session: Any = None) -> SnowflakeClient:
    """Return a client: explicit ``session`` > active Snowpark session > connector.

    For the Snowpark paths the session is pointed at ``cfg.snowflake.database`` /
    ``.schema`` (``DATAMART_NEXT`` / ``NEXTZEN`` by default).
    """
    db, schema = cfg.snowflake.database, cfg.snowflake.schema
    if session is not None:
        return SnowparkClient(session, db, schema)
    try:
        from snowflake.snowpark.context import get_active_session

        return SnowparkClient(get_active_session(), db, schema)
    except Exception:
        return ConnectorClient(cfg)


# ---------------------------------------------------------------------------
# Query building + row mapping (ported)
# ---------------------------------------------------------------------------


def _metadata_sql(db: str, schema: str, table_name: str, group_override: str) -> str:
    _assert_safe_table_name(table_name)
    if group_override:
        return f"""
            SELECT DISTINCT '{group_override}' as group_name, f.field_id, f.field_label, f.field_type
            FROM DATAMART_NEXT.NEXTZEN.{table_name} f
            WHERE f.organization_id = ?
        """
    return f"""
        SELECT DISTINCT p.name as group_name, f.field_id, f.field_label, f.field_type
        FROM DATAMART_NEXT.NEXTZEN.{table_name} f
        left join DATAMART_NEXT.next.program p on f.program_id = p.id
        WHERE f.organization_id = ?
    """


def _map_row_to_intake_field(r: Dict[str, Any]) -> IntakeFieldInput:
    def pick(*keys: str) -> Any:
        for k in keys:
            if k in r and r[k] is not None:
                return r[k]
        # allow None to pass through when the key exists but is null
        for k in keys:
            if k in r:
                return r[k]
        return None

    _id = pick("FIELD_ID", "field_id")
    label = pick("FIELD_LABEL", "field_label")
    ftype = pick("FIELD_TYPE", "field_type")
    group = pick("GROUP_NAME", "group_name")
    return IntakeFieldInput(
        field_id=str(_id if _id is not None else ""),
        field_label=str(label if label is not None else ""),
        field_type=None if ftype is None else str(ftype),
        group_name=None if group is None else str(group),
    )


def fetch_field_metadata_for_org(
    client: SnowflakeClient,
    cfg: AppConfig,
    organization_id: str,
    table_name: str,
    group_override: str,
) -> List[IntakeFieldInput]:
    """Load field metadata for an org from a single Snowflake table.

    Rows with an empty ``field_id`` are dropped (matches the TS behavior).
    """
    _assert_safe_table_name(table_name)
    sf = cfg.snowflake
    sql = _metadata_sql(sf.database, sf.schema, table_name, group_override)
    rows = client.query(sql, [organization_id])
    mapped = [_map_row_to_intake_field(r) for r in rows]
    return [r for r in mapped if len(r.field_id) > 0]

### Sweep orchestration
`run_sweep` is the entry point: resolve target orgs, and for each — fetch fields, export the template, merge, resolve create-vs-update, then print (dry run) or import the **"Custom Model"**.

In [ ]:
# --- sweep ---
#!/usr/bin/env python
"""ThoughtSpot "Sweeper" (simplified, print-only).

For each target Submittable organization:
  1. Pull intake field metadata from Snowflake.
  2. Mint a ThoughtSpot bearer token scoped to that org.
  3. Export the read-only template model (by name) from that org.
  4. Merge each field as a formula+column into the template TML.
  5. Look up the org's existing "Custom Model" by name (create vs update).
  6. Import the merged TML back into ThoughtSpot as "Custom Model".

Everything is printed to stdout (shows directly in a Snowflake Notebook cell):
progress lines plus the full merged TML. No files are written -- no logs, no
support bundles, no ``.model.tml`` artifact.

Target selection precedence:
  org_ids_file  -> newline-separated Submittable org ids
  org_ids       -> a Python list of ids (notebook convenience)
  next_org_id   -> a single Submittable org id
  none of them  -> every TS org whose name starts with "Next-"
"""


import argparse
import json
import os
import sys
import traceback
from dataclasses import dataclass
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Sequence


# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------

# Display name for every per-org custom model. Each org has exactly one.
CUSTOM_ORG_MODEL_NAME = "Custom Model"

# Stable obj_id for every per-org custom model (ThoughtSpot scopes obj_id per org).
CUSTOM_ORG_MODEL_OBJ_ID = "next_custom_org_model"

# (fields-table, responses-table) tuples to sweep, in order. `group_override`
# specifies how to group in TS; when "", the field's program is used.
FIELD_TABLE_PAIRS: Sequence[Dict[str, str]] = (
    {"fields_table_name": "INTAKE_FORM_FIELDS_PUBLISHED", "responses_table_name": "ROUND_FORM_RESPONSE", "group_override": ""},
    {"fields_table_name": "REVIEW_FORM_FIELDS", "responses_table_name": "ROUND_FORM_RESPONSE", "group_override": ""},
    {"fields_table_name": "PROFILE_FORM_FIELDS", "responses_table_name": "PROFILE_COMPLIANCE", "group_override": "Profile Responses"},
    {"fields_table_name": "PROGRESS_FORM_FIELDS", "responses_table_name": "AWARD_PROGRESS_REPORT", "group_override": ""},
)


def _log(msg: str) -> None:
    """Timestamped print -- the only logging. Shows in the notebook cell output."""
    print(f"{datetime.now(timezone.utc).isoformat()} {msg}")


# ---------------------------------------------------------------------------
# TML doc prep
# ---------------------------------------------------------------------------


def _is_record(x: Any) -> bool:
    return isinstance(x, dict)


def _enforce_dont_index_default(model: Dict[str, Any]) -> None:
    """Default every column to DONT_INDEX unless the template sets index_type."""
    if not isinstance(model.get("columns"), list):
        return
    for col in model["columns"]:
        if not _is_record(col):
            continue
        props = col["properties"] if _is_record(col.get("properties")) else {}
        if "index_type" not in props:
            col["properties"] = {**props, "index_type": "DONT_INDEX"}


def _prepare_tml_for_import(doc: Dict[str, Any], existing_guid: Optional[str]) -> None:
    """Set model name/obj_id, drop or reuse guid, apply the DONT_INDEX default."""
    if _is_record(doc.get("model")):
        doc["model"]["name"] = CUSTOM_ORG_MODEL_NAME
        doc["model"]["obj_id"] = CUSTOM_ORG_MODEL_OBJ_ID
        _enforce_dont_index_default(doc["model"])
    doc["obj_id"] = CUSTOM_ORG_MODEL_OBJ_ID
    if existing_guid:
        doc["guid"] = existing_guid
    else:
        doc.pop("guid", None)


# ---------------------------------------------------------------------------
# ThoughtSpot tokens
# ---------------------------------------------------------------------------


def _require_username_and_secret(cfg: AppConfig):
    ts = cfg.thoughtspot
    if not ts.username or not ts.secret_key:
        raise RuntimeError(
            "Cannot mint tokens: set THOUGHTSPOT_USERNAME and THOUGHTSPOT_SECRET_KEY "
            "(or set THOUGHTSPOT_BEARER_TOKEN to bypass)."
        )
    return ts.username, ts.secret_key


def _mint_discovery_token(cfg: AppConfig) -> str:
    """Token scoped to the primary org (org_id=0) for org discovery."""
    if cfg.thoughtspot.bearer_token:
        _log("Using THOUGHTSPOT_BEARER_TOKEN for all phases.")
        return cfg.thoughtspot.bearer_token
    username, secret_key = _require_username_and_secret(cfg)
    return mint_full_bearer_token(cfg.thoughtspot.host, username, secret_key, 0)


def _mint_import_token(cfg: AppConfig, thoughtspot_org_id: int) -> str:
    """Token scoped to one target org (used for both template export and import)."""
    if cfg.thoughtspot.bearer_token:
        return cfg.thoughtspot.bearer_token
    username, secret_key = _require_username_and_secret(cfg)
    return mint_full_bearer_token(cfg.thoughtspot.host, username, secret_key, thoughtspot_org_id)


# ---------------------------------------------------------------------------
# Target selection
# ---------------------------------------------------------------------------


@dataclass
class SweepTarget:
    next_org_id: str
    thoughtspot_org_id: int


@dataclass
class CliOptions:
    next_org_id: Optional[str] = None
    org_ids_file: Optional[str] = None
    org_ids: Optional[List[str]] = None
    dry_run: bool = False
    model_name: Optional[str] = None


def _read_org_ids_file(file_path: str) -> List[str]:
    with open(file_path, "r", encoding="utf-8") as f:
        raw = f.read()
    out: List[str] = []
    seen = set()
    for line in raw.splitlines():
        _id = line.strip()
        if not _id or _id.startswith("#") or _id in seen:
            continue
        seen.add(_id)
        out.append(_id)
    return out


def _targets_from_id_list(cfg: AppConfig, ids: List[str], discovery_token: str) -> List[SweepTarget]:
    if not ids:
        return []
    orgs = list_orgs(cfg.thoughtspot.host, discovery_token)
    ts_org_id_by_name = {o.name: o.org_id for o in orgs}
    targets: List[SweepTarget] = []
    for next_org_id in ids:
        ts_org_id = ts_org_id_by_name.get(f"Next-{next_org_id}")
        if ts_org_id is None:
            _log(f'No ThoughtSpot org named "Next-{next_org_id}"; skipping.')
            continue
        targets.append(SweepTarget(next_org_id=next_org_id, thoughtspot_org_id=ts_org_id))
    _log(f"Resolved {len(targets)} of {len(ids)} org id(s) to ThoughtSpot orgs.")
    return targets


def _resolve_sweep_targets(cfg: AppConfig, opts: CliOptions, discovery_token: str) -> List[SweepTarget]:
    host = cfg.thoughtspot.host

    # 1. org_ids_file: explicit list of Submittable org ids.
    if opts.org_ids_file:
        if opts.next_org_id:
            _log("Both org_ids_file and next_org_id were provided; org_ids_file wins.")
        ids = _read_org_ids_file(opts.org_ids_file)
        _log(f"Loaded {len(ids)} org id(s) from {os.path.abspath(opts.org_ids_file)}.")
        return _targets_from_id_list(cfg, ids, discovery_token)

    # 1b. org_ids: in-memory list (notebook convenience).
    if opts.org_ids:
        seen: set = set()
        ids = []
        for _id in opts.org_ids:
            _id = str(_id).strip()
            if _id and _id not in seen:
                seen.add(_id)
                ids.append(_id)
        _log(f"Received {len(ids)} org id(s) from the in-memory list.")
        return _targets_from_id_list(cfg, ids, discovery_token)

    # 2. next_org_id: single org.
    if opts.next_org_id:
        name = f"Next-{opts.next_org_id}"
        ts_org_id = find_org_id_by_name(host, discovery_token, name)
        if ts_org_id is None:
            raise RuntimeError(
                f'No ThoughtSpot org found whose name equals "{name}". '
                "The TS org name must be Next-{Submittable organization_id}."
            )
        return [SweepTarget(next_org_id=opts.next_org_id, thoughtspot_org_id=ts_org_id)]

    # 3. No selection: every TS org named "Next-*".
    orgs = list_orgs(host, discovery_token)
    targets: List[SweepTarget] = []
    for o in orgs:
        if not o.name.startswith("Next-"):
            continue
        next_org_id = o.name[len("Next-"):]
        if next_org_id:
            targets.append(SweepTarget(next_org_id=next_org_id, thoughtspot_org_id=o.org_id))
    _log(f"Discovered {len(targets)} ThoughtSpot org(s) to sweep (of {len(orgs)} total).")
    return targets


# ---------------------------------------------------------------------------
# Per-org sweep
# ---------------------------------------------------------------------------


@dataclass
class SweepOutcome:
    status: Any  # int | f"Error {n}" | "DRY_RUN" | "ERROR"
    ok: bool


def _sweep_one_org(
    cfg: AppConfig,
    opts: CliOptions,
    target: SweepTarget,
    sf_client: Optional[SnowflakeClient],
    prefetched: Optional[List[tuple]] = None,
) -> SweepOutcome:
    org = target.next_org_id
    host = cfg.thoughtspot.host

    try:
        _log(f"[{org}] starting (ts-org={target.thoughtspot_org_id})")

        # 1. Field metadata as a list of (responses_table_name, fields).
        #    Either supplied by the caller (notebook ran the SQL and passed the
        #    variable in) or fetched here via the Snowflake client.
        field_results: List[tuple] = []
        if prefetched is not None:
            field_results = list(prefetched)
            for resp_table, fields in field_results:
                _log(f"[{org}] using {len(fields)} prefetched field(s) for {resp_table}")
        else:
            for pair in FIELD_TABLE_PAIRS:
                fields = fetch_field_metadata_for_org(
                    sf_client, cfg, org, pair["fields_table_name"], pair["group_override"]
                )
                _log(f"[{org}] fetched {len(fields)} field(s) from {pair['fields_table_name']}")
                field_results.append((pair["responses_table_name"], fields))

        # 2. Per-org token (template export + import).
        import_token = _mint_import_token(cfg, target.thoughtspot_org_id)

        # 3. Export the template TML from this org by name.
        template_name = (opts.model_name or cfg.tml.model_name or "").strip()
        if not template_name:
            raise RuntimeError("Missing template model: set TML_MODEL_NAME or pass model_name.")
        base_yaml = fetch_model_tml_yaml_by_name(host, import_token, template_name)

        # 4. Parse + merge in the fields, each bound to its responses table.
        doc = parse_tml(base_yaml)
        if not _is_record(doc):
            raise RuntimeError("Base model YAML must parse to a mapping at the root.")
        for resp_table, fields in field_results:
            if fields:
                doc = merge_intake_formulas_into_model_tml(doc, fields, resp_table)

        # 5. Resolve create vs update.
        existing = find_existing_model_by_name_if_unique(
            host, import_token, CUSTOM_ORG_MODEL_NAME, expected_obj_id=CUSTOM_ORG_MODEL_OBJ_ID
        )
        existing_guid = existing["guid"] if existing else None
        _log(
            f'[{org}] {"upsert existing " + existing_guid if existing_guid else "create new"} '
            f'"{CUSTOM_ORG_MODEL_NAME}"'
        )

        # 6. Finalize the merged TML. On a dry run, print it for inspection;
        #    on a real run just note its size (the import happens below).
        _prepare_tml_for_import(doc, existing_guid)
        yaml = serialize_tml(doc)
        if opts.dry_run:
            _log(f"[{org}] merged TML ready ({len(yaml.splitlines())} lines); dry run: skipping import")
            return SweepOutcome("DRY_RUN", True)
        _log(f"[{org}] merged TML ready ({len(yaml.splitlines())} lines); importing")

        # 7. Import.
        create_new = not existing_guid
        result = import_table_tml(
            host=host,
            bearer_token=import_token,
            tml_yaml=yaml,
            create_new=create_new,
            import_policy=cfg.thoughtspot.import_policy,
            timeout_s=cfg.tml.import_timeout_s,
        )

        # 8. Interpret the response.
        if not is_thoughtspot_import_response_ok(result.status, result.data):
            _log(f"[{org}] import FAILED (HTTP {result.status}): {json.dumps(result.data, default=str)[:1000]}")
            code = extract_import_error_code(result.data)
            return SweepOutcome(f"Error {code}" if code is not None else result.status, False)

        guid = result.guid or existing_guid
        if not guid:
            _log(f"[{org}] import returned HTTP {result.status} but no GUID; model likely not created")
            return SweepOutcome(result.status, False)

        _log(f"[{org}] import completed (guid={guid})")
        return SweepOutcome(result.status, True)
    except Exception as err:
        _log(f"[{org}] ERROR: {err}")
        return SweepOutcome("ERROR", False)


# ---------------------------------------------------------------------------
# Entry points
# ---------------------------------------------------------------------------


def run_sweep(
    *,
    next_org_id: Optional[str] = None,
    org_ids_file: Optional[str] = None,
    org_ids: Optional[List[str]] = None,
    dry_run: bool = False,
    model_name: Optional[str] = None,
    cfg: Optional[AppConfig] = None,
    session: Any = None,
    prefetched_field_results: Optional[List[tuple]] = None,
) -> int:
    """Run the sweep. Returns 0 on success, 1 if any org failed.

    In a Snowflake Notebook (with the ts_sweeper folder on sys.path)::

        from sweep import run_sweep

        run_sweep(org_ids=["123", "456"])   # creds from settings.py, session auto-pulled

    ``prefetched_field_results`` lets you run the field SQL yourself (e.g. in a
    notebook cell) and pass the result in as a variable, skipping the built-in
    Snowflake fetch. Format: ``[(responses_table_name, [IntakeFieldInput, ...]), ...]``.
    Intended for a single target org.
    """
    opts = CliOptions(
        next_org_id=next_org_id,
        org_ids_file=org_ids_file,
        org_ids=org_ids,
        dry_run=dry_run,
        model_name=model_name,
    )
    cfg = cfg or load_config()

    try:
        # Only need a Snowflake client when we're the ones fetching.
        sf_client = None if prefetched_field_results is not None else get_default_client(cfg, session)
        discovery_token = _mint_discovery_token(cfg)
        targets = _resolve_sweep_targets(cfg, opts, discovery_token)
    except Exception as err:
        _log(f"Top-level failure: {err}")
        traceback.print_exc()
        return 1

    if not targets:
        _log("No ThoughtSpot orgs found to sweep.")
        return 1

    total = len(targets)
    failed = 0
    for i, target in enumerate(targets, 1):
        _log(f"Began sweep on {target.next_org_id} ({i} out of {total})")
        outcome = _sweep_one_org(cfg, opts, target, sf_client, prefetched=prefetched_field_results)
        _log(f"Finished run on {target.next_org_id} ({i} out of {total}). Status {outcome.status}")
        if not outcome.ok:
            failed += 1

    if failed:
        _log(f"Sweep finished with {failed} failure(s) out of {total} org(s).")
        return 1
    _log(f"Sweep finished cleanly across {total} org(s).")
    return 0

## 3. Config
Credentials live in **`sweep_config.py`** (a separate file — keeps your secret out of the notebook). Edit it, upload it alongside the notebook, and this cell imports it.

In [ ]:
from snowflake.snowpark.context import get_active_session
from sweep_config import CONFIG      # <-- edit sweep_config.py with your creds

session = get_active_session()
cfg = config_from_mapping(CONFIG)
print("config OK; host:", cfg.thoughtspot.host)

## 4. Run
`DRY_RUN=True` builds without importing; `False` imports. Logs stream to the cell and are written to `EAS.INTEGRATION.SWEEP_LOG` — one row per line with:

- **`ORGANIZATION_ID`** — for per-org aggregation (null on run-level lines)
- **`LOG_TYPE`** — `data` / `query` / `snowflake_connection` / `tml` / `export` / `import` / `error` / `info`
- plus `RUN_ID`, `RUN_STARTED_AT`, `SEQ`, `MESSAGE`

The write is wrapped in try/except so a logging failure won't hide the sweep result.

In [ ]:
import io, sys, uuid, re, contextlib
from datetime import datetime, timezone

ORG_IDS   = ["rj5iopIB-4lsgZs-srsLu9_IPTvw-1tuB_El"]   # one or more Submittable org ids
DRY_RUN   = True
LOG_TABLE = "EAS.INTEGRATION.SWEEP_LOG"                 # DB.SCHEMA.TABLE for the logs

def _type_of(line):
    s = line.lower()
    if any(w in s for w in ("error", "failed", "not authorized", "does not exist", "aborted", "traceback")):
        return "error"
    if "sql:" in s or re.search(r"\bselect ", s):
        return "query"
    if "fetched" in s and "field" in s:
        return "data"
    if "importing" in s or "import completed" in s or "import accepted" in s or "task_id" in s:
        return "import"
    if "merged tml" in s or "field types" in s or "dry run" in s or "skipping import" in s:
        return "tml"
    if "export" in s or "template" in s or "upsert existing" in s or "create new" in s:
        return "export"
    if "import" in s:
        return "import"
    return "info"

_orig_stdout = sys.stdout
class _Tee(io.TextIOBase):
    def __init__(self):
        self.buf = []
    def write(self, s):
        _orig_stdout.write(s)
        self.buf.append(s)
        return len(s)

run_id  = uuid.uuid4().hex
started = datetime.now(timezone.utc).isoformat()

# Run each org on its own so EVERY captured line is tagged with the org id we
# passed in -- no parsing, no null organization_id.
rows, seq, exit_code = [], 0, 0
for org_id in ORG_IDS:
    _tee = _Tee()
    with contextlib.redirect_stdout(_tee):
        rc = run_sweep(org_ids=[org_id], dry_run=DRY_RUN, cfg=cfg, session=session)
    exit_code = exit_code or rc
    for l in "".join(_tee.buf).splitlines():
        if l.strip():
            rows.append((run_id, started, seq, org_id, _type_of(l), l))
            seq += 1

if rows:
    try:
        (session.create_dataframe(
             rows, schema=["RUN_ID", "RUN_STARTED_AT", "SEQ", "ORGANIZATION_ID", "LOG_TYPE", "MESSAGE"])
                .write.mode("append").save_as_table(LOG_TABLE.split("."), column_order="name"))
        print(f"wrote {len(rows)} log rows to {LOG_TABLE}")
    except Exception as _e:
        print(f"WARN: could not write logs to {LOG_TABLE}: {_e}")

print(f"exit_code: {exit_code} (run_id {run_id})")